# 🌱 Irrigation Need Prediction
### Kaggle Playground Series S6E4 — LightGBM + Advanced Feature Engineering

**Pipeline:**
- Advanced agricultural feature engineering (10 kategori)
- OrdinalEncoder + StratifiedKFold cross-validation
- LightGBM multiclass classifier
- Balanced Accuracy metriği
- Model export (joblib) → Hugging Face deploy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from collections import defaultdict
from matplotlib.lines import Line2D

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    LabelEncoder,
    OrdinalEncoder
)

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

from lightgbm import LGBMClassifier, log_evaluation

print('✅ Tüm kütüphaneler yüklendi.')

In [ ]:
# Renk paleti
palette = sns.color_palette('bright', 10)

## 📂 1. Veri Yükleme

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/playground-series-s6e4/test.csv')

print(f'Train: {train.shape} | Test: {test.shape}')
train.head()

## 🔍 2. Keşifsel Veri Analizi (EDA)

In [ ]:
cat_columns     = train.select_dtypes(exclude=np.number).columns
numerical_cols  = train.select_dtypes(include='number').columns.difference(['id'])

print('Kategorik sütunlar:', cat_columns.tolist())
print('Sayısal sütunlar :', numerical_cols.tolist())

In [ ]:
# Kategorik dağılım karşılaştırması (train vs test)
for col in cat_columns:
    print('='*80)
    print(f'### {col}')
    print('Train:')
    print(train[col].value_counts(normalize=True).round(3))
    if col in test.columns:
        print('Test:')
        print(test[col].value_counts(normalize=True).round(3))

In [ ]:
# Sayısal dağılım grafikleri (KDE + Boxplot)
num_cols_grid = 3
num_rows_grid = (len(numerical_cols) + num_cols_grid - 1) // num_cols_grid

fig, axes = plt.subplots(num_rows_grid, num_cols_grid,
                         figsize=(15, 5 * num_rows_grid),
                         constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.kdeplot(train[col], ax=axes[i], color=palette[0], fill=True, label='Train')
    if col in test.columns:
        sns.kdeplot(test[col], ax=axes[i], color=palette[1], fill=True, label='Test')
    axes[i].set_title(col)

    ax_box = axes[i].inset_axes([0.2, -0.4, 0.6, 0.2])
    sns.boxplot(x=train[col], ax=ax_box, orient='h')
    ax_box.set(xlabel='')

for j in range(len(numerical_cols), len(axes)):
    fig.delaxes(axes[j])

custom_lines = [
    Line2D([0], [0], color=palette[0], lw=4, label='Train'),
    Line2D([0], [0], color=palette[1], lw=4, label='Test')
]
fig.legend(handles=custom_lines, loc='upper right',
           bbox_to_anchor=(1, 1), frameon=False)
plt.show()

In [ ]:
# Korelasyon matrisi
target_cols = numerical_cols.difference(['Irrigation_Need'])
corr_matrix = train[target_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Korelasyon Matrisi')
plt.tight_layout()
plt.show()

## ⚙️ 3. Feature Engineering

In [ ]:
def create_advanced_agri_features(df):
    """
    Tarımsal domain bilgisi ile zenginleştirilmiş feature seti.
    10 kategori: İklim, Su Dengesi, Toprak, Bitki Evresi,
    Kimya, Malçlama, Çevresel Kombinasyon, Gelişmiş Etkileşimler.
    """
    df = df.copy()

    # 1. TEMEL TEMİZLİK
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

    # 2. İKLİM ÖZELLİKLERİ
    df['evapotranspiration_index'] = (df['Temperature_C'] * (100 - df['Humidity'])) / 100
    df['heat_stress']  = np.maximum(0, df['Temperature_C'] - 30)
    df['cool_stress']  = np.maximum(0, 10 - df['Temperature_C'])
    df['wind_effect']  = df['Wind_Speed_kmh'] * (100 - df['Humidity']) / 100
    df['thermal_wind'] = df['Temperature_C'] * (1 + df['Wind_Speed_kmh'] / 100)
    df['sun_stress']   = df['Sunlight_Hours'] * df['Temperature_C']

    # 3. SU DENGESİ (Leakage-Safe)
    if 'Previous_Irrigation_mm' in df.columns:
        df['total_water_input'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    else:
        df['total_water_input'] = df['Rainfall_mm']

    df['water_deficit'] = df['evapotranspiration_index'] - df['total_water_input']
    df['drought_index'] = df['Temperature_C'] / (df['total_water_input'] + 1)

    # 4. TOPRAK MÜHENDİSLİĞİ
    soil_retention = {'Clay': 1.2, 'Silt': 1.0, 'Loamy': 0.9, 'Sandy': 0.6}
    median_retention = np.nanmedian(list(soil_retention.values()))
    df['soil_retention_factor'] = df['Soil_Type'].map(soil_retention).fillna(median_retention)
    df['effective_soil_moisture'] = df['Soil_Moisture'] * df['soil_retention_factor']
    df['soil_water_buffer']       = df['effective_soil_moisture'] / (df['water_deficit'] + 1)

    # 5. BİTKİ EVRESİ
    stage_map = {'Sowing': 1, 'Vegetative': 2, 'Flowering': 3, 'Harvest': 4}
    df['stage_numeric']          = df['Crop_Growth_Stage'].map(stage_map).fillna(0)
    df['is_critical_stage']      = (df['Crop_Growth_Stage'] == 'Flowering').astype(int)
    df['growth_temp_interaction'] = df['stage_numeric'] * df['Temperature_C']
    df['critical_water_need']    = df['is_critical_stage'] * (df['Temperature_C'] / (df['Soil_Moisture'] + 1))

    # 6. TOPRAK KİMYASI
    df['optimal_ph']    = df['Soil_pH'].between(6.0, 7.0).astype(int)
    df['nutrient_index'] = df['Organic_Carbon'] * df['optimal_ph']
    df['ph_deviation']  = (df['Soil_pH'] - 6.5).abs()

    # 7. MALÇLAMA ETKİSİ
    df['mulch_binary']           = df['Mulching_Used'].map({'Yes': 1, 'No': 0}).fillna(0)
    df['sun_exposure_adjusted']  = df['Sunlight_Hours'] / (1 + df['mulch_binary'])
    df['evaporation_control']    = df['mulch_binary'] * df['Humidity']

    # 8. ÇEVRESEL KOMBİNASYON (Hash encoding)
    df['env_key']      = df['Region'].astype(str) + '_' + df['Season'].astype(str) + '_' + df['Soil_Type'].astype(str)
    df['env_key_code'] = df['env_key'].astype('category').cat.codes

    # 9. GELİŞMİŞ ETKİLEŞİMLER
    df['temp_moisture_ratio'] = df['Temperature_C'] / (df['Soil_Moisture'] + 1)
    df['water_efficiency']    = df['effective_soil_moisture'] / (df['total_water_input'] + 1)
    df['climate_severity']    = df['heat_stress'] + df['wind_effect'] + df['water_deficit']

    # 10. SON TEMİZLİK
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df

train = create_advanced_agri_features(train)
test  = create_advanced_agri_features(test)

print(f'✅ Feature engineering tamamlandı. Train: {train.shape} | Test: {test.shape}')

## 🔧 4. Ön İşleme (Preprocessing)

In [ ]:
# Hedef değişkeni ve ID'yi ayır
TARGET = 'Irrigation_Need'
DROP_COLS = ['id', TARGET, 'env_key']  # env_key ham string, kodlanmış hali env_key_code

X = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
y_raw = train[TARGET]

X_test = test.drop(columns=[c for c in ['id', 'env_key'] if c in test.columns])

# Sütun tutarlılığı garantisi
X_test = X_test[X.columns]

print(f'X: {X.shape} | X_test: {X_test.shape}')
print(f'Sütun eşleşmesi: {list(X.columns) == list(X_test.columns)}')

In [ ]:
# Hedef LabelEncoder
le_target = LabelEncoder()
y = pd.Series(le_target.fit_transform(y_raw), name=TARGET)

print('Sınıf eşlemesi:')
for i, cls in enumerate(le_target.classes_):
    print(f'  {i} → {cls}')

In [ ]:
# Tüm sütunları OrdinalEncoder ile encode et
# (Kategorik + sayısal — LightGBM ikisini de kabul eder)
feature_columns = X.columns.tolist()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

preprocessor = ColumnTransformer([
    ('enc', encoder, feature_columns)
], remainder='passthrough')

pipe = Pipeline([('preprocessor', preprocessor)])

# FIT sadece train üzerinde (leakage önlemi)
X_proc      = pd.DataFrame(pipe.fit_transform(X),      columns=feature_columns)
X_test_proc = pd.DataFrame(pipe.transform(X_test),     columns=feature_columns)

print(f'✅ Ön işleme tamamlandı. X_proc: {X_proc.shape}')

## 🚀 5. Model Eğitimi — LightGBM + StratifiedKFold CV

In [ ]:
N_SPLITS   = 5
N_CLASSES  = len(le_target.classes_)

# StratifiedKFold — eksik olan kısım burada tanımlanıyordu!
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# GPU kaldırıldı (Hugging Face CPU ortamıyla uyumlu)
LGBM_PARAMS = {
    'learning_rate'         : 0.018,
    'num_leaves'            : 42,
    'max_depth'             : 9,
    'min_child_samples'     : 34,
    'feature_fraction'      : 0.6,
    'feature_fraction_bynode': 0.8,
    'bagging_fraction'      : 0.8,
    'bagging_freq'          : 5,
    'lambda_l1'             : 1e-6,
    'lambda_l2'             : 0.3,
    'min_gain_to_split'     : 0.01,
    'num_class'             : N_CLASSES,
    'objective'             : 'multiclass',
    'metric'                : 'multi_logloss',
    'n_estimators'          : 5000,
    'early_stopping_rounds' : 100,
    'class_weight'          : 'balanced',
    'random_state'          : 42,
    'verbosity'             : -1,
    'boosting_type'         : 'gbdt',
    'n_jobs'                : -1,
}

In [ ]:
scores      = []
oof_preds   = np.zeros((len(X_proc), N_CLASSES))
test_preds  = np.zeros((len(X_test_proc), N_CLASSES))
importances = []
models      = []  # Her fold modeli sakla (ensemble için)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_proc, y)):
    print(f'\n🔥 Fold {fold + 1}/{N_SPLITS}')

    X_tr, X_val = X_proc.iloc[train_idx], X_proc.iloc[valid_idx]
    y_tr, y_val = y.iloc[train_idx],       y.iloc[valid_idx]

    model = LGBMClassifier(**LGBM_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[log_evaluation(period=0)]
    )

    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)

    score = balanced_accuracy_score(y_val, y_pred)
    print(f'  Balanced Accuracy: {score:.5f}')
    scores.append(score)

    oof_preds[valid_idx]  = y_proba
    test_preds           += model.predict_proba(X_test_proc) / N_SPLITS
    importances.append(model.feature_importances_)
    models.append(model)

print('\n' + '='*50)
print(f'CV Mean : {np.mean(scores):.5f}')
print(f'CV Std  : {np.std(scores):.5f}')

## 📊 6. OOF Sonuçları & Feature Importance

In [ ]:
# OOF overall skoru
oof_labels = np.argmax(oof_preds, axis=1)
oof_score  = balanced_accuracy_score(y, oof_labels)
print(f'OOF Balanced Accuracy: {oof_score:.5f}')
print()
print(classification_report(y, oof_labels, target_names=le_target.classes_))

In [ ]:
# Feature importance grafiği
mean_importance = np.mean(importances, axis=0)
feat_imp_df = pd.DataFrame({
    'feature'   : feature_columns,
    'importance': mean_importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 12))
sns.barplot(data=feat_imp_df.head(30), x='importance', y='feature', palette='viridis')
plt.title('Top 30 Feature Importance (Ortalama)')
plt.tight_layout()
plt.show()

## 📤 7. Submission

In [ ]:
final_labels      = np.argmax(test_preds, axis=1)
final_predictions = le_target.inverse_transform(final_labels)

submission = pd.DataFrame({
    'id'             : test['id'],
    'Irrigation_Need': final_predictions
})
submission.to_csv('submission.csv', index=False)
print('✅ submission.csv kaydedildi.')
submission.head(10)

## 💾 8. Model Kayıt (Hugging Face için)

In [ ]:
# Son fold modeli yerine en iyi fold modelini kaydet
best_fold  = int(np.argmax(scores))
best_model = models[best_fold]
print(f'En iyi fold: {best_fold + 1} | Score: {scores[best_fold]:.5f}')

joblib.dump(best_model,  'lgbm_model.pkl')
joblib.dump(pipe,        'preprocessor.pkl')
joblib.dump(le_target,   'label_encoder.pkl')
joblib.dump(feature_columns, 'feature_columns.pkl')  # Sütun sırası kritik!

print('✅ Kaydedilen dosyalar:')
print('   lgbm_model.pkl      — Eğitilmiş LightGBM modeli')
print('   preprocessor.pkl    — OrdinalEncoder pipeline')
print('   label_encoder.pkl   — Hedef değişken encoder')
print('   feature_columns.pkl — Sütun sıra listesi')